Perform forward sequence alignment of sequencing-derived fragments against the template plasmid

Generate a CSV file reporting, for each FASTA entry:
- Sequence ID
- Sequence length
- Alignment identity
- Mapping status (identity ≥ 95%)
- Start and end coordinates on the plasmid

Extract fragments mapped to the plasmid and export them as a FASTA file

In [ ]:
from Bio import SeqIO
from Bio.Align import PairwiseAligner
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import sys

In [ ]:
fasta_file = "fragments.fasta"      
plasmid_file = "plasmid.fasta"      
output_csv = "match_results.csv"

In [ ]:
plasmid_record = next(SeqIO.parse(plasmid_file, "fasta"))
plasmid_seq = str(plasmid_record.seq).upper()

aligner = PairwiseAligner()
aligner.mode = 'local' 
aligner.match_score = 1 
aligner.mismatch_score = -1
aligner.open_gap_score = -2
aligner.extend_gap_score = -0.5

records = list(SeqIO.parse(fasta_file, "fasta"))
results = []
total = len(records)

print(f"共有 {total} 条序列待检测...\n")

def compute_alignment(seq1, seq2):
    """返回最佳比对的identity、起始和终止位点"""
    alignments = aligner.align(seq1, seq2)
    best = alignments[0]
    aligned_plasmid, aligned_seq = best.aligned

    if len(aligned_plasmid) == 0 or len(aligned_seq) == 0:
        return 0.0, None, None

    start_p, end_p = aligned_plasmid[0]
    start_s, end_s = aligned_seq[0]

    sub_p = seq1[start_p:end_p]
    sub_s = seq2[start_s:end_s]
    min_len = min(len(sub_p), len(sub_s))
    if min_len == 0:
        return 0.0, start_p, end_p

    matches = sum(sub_p[i] == sub_s[i] for i in range(min_len))
    identity = matches / min_len
    return identity, start_p + 1, end_p  

for i, record in enumerate(records, start=1):
    seq = str(record.seq).upper()
    identity, start, end = compute_alignment(plasmid_seq, seq)
    match = "Yes" if identity >= 0.95 else "No"

    results.append({
        "Sequence_ID": record.id,
        "Length": len(seq),
        "Identity": round(identity, 3),
        "Start": start if start else "",
        "End": end if end else "",
        "Match": match
    })

    sys.stdout.write(f"\r比对进度: {i}/{total} ({i/total*100:.1f}%)")
    sys.stdout.flush()

print("\n比对完成！\n")

df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)

print(f"结果已保存到: {output_csv}")

In the generated CSV file, manually verify whether the sequence lengthmatches (end position - start position + 1) using an Excel function
Add a new column named "lengthMatch" and use the formula: =IF(E2-D2+1=B2,"Yes","No")，then perform secondary filtering based on length consistency

In [ ]:
df = pd.read_csv("match_results.csv")

filtered = df[(df["Match"] == "Yes") & (df["lengthmatch"] == "Yes")]

filtered = filtered[["Sequence_ID", "Length", "Identity", "Start", "End", "Match", "lengthmatch"]]

filtered.to_csv("match&length.csv", index=False)

print(f"筛选完成，共 {len(filtered)} 条匹配序列，结果已保存为match&length.csv")

 Identify plasmid fragments that can be concatenated, and determine the number and positions of break sites

In [ ]:
df = pd.read_csv('match&length.csv')

df_filtered = df[(df['Match'] == 'Yes') & (df['lengthmatch'] == 'Yes')]

df_filtered = df_filtered.sort_values(by='Start').reset_index(drop=True)

connected_pairs = []

total = len(df_filtered)
print("开始查找首尾相连序列...")

for i, row1 in df_filtered.iterrows():
    if i % 10 == 0 or i == total - 1:  
        print(f"进度: {i+1}/{total} ({(i+1)/total*100:.1f}%)")
    for j, row2 in df_filtered.iterrows():
        if i >= j:
            continue
        if row1['End'] + 1 == row2['Start']:
            connected_pairs.append({
                'Seq1_ID': row1['Sequence_ID'],
                'Seq1_Start': row1['Start'],
                'Seq1_End': row1['End'],
                'Seq2_ID': row2['Sequence_ID'],
                'Seq2_Start': row2['Start'],
                'Seq2_End': row2['End']
            })

output_df = pd.DataFrame(connected_pairs)
output_df.to_csv('connected_pairs.csv', index=False)

print(f"共找到 {len(connected_pairs)} 组首尾相连序列，已保存到 connected_pairs.csv")

Analyze break site positions and count the frequency of each site

In [ ]:
input_csv = 'connected_pairs.csv'    
bar_csv_out = 'end_position_frequency.csv'  

df = pd.read_csv(input_csv)

unique_ends = df['Seq1_End'].drop_duplicates()
end_counts = Counter(df['Seq1_End'])

bar_data = pd.DataFrame({'Seq1_End': list(end_counts.keys()),
                         'Frequency': list(end_counts.values())})
bar_data.to_csv(bar_csv_out, index=False)
print(f"柱状图数据已保存为 {bar_csv_out}")

plt.figure(figsize=(10,6))
plt.bar(bar_data['Seq1_End'], bar_data['Frequency'], color='skyblue')
plt.xlabel('Seq1_End position')
plt.ylabel('Frequency')
plt.title('Frequency of Seq1_End positions')
plt.tight_layout()
plt.savefig('end_position_frequency.png', dpi=300)
plt.show()

print("End 位置频率柱状图已保存为 end_position_frequency.png")

Extract FASTA sequences corresponding to entries passing the secondary filtering in the CSV file

In [ ]:
fragments_fasta = 'fragments.fasta'  
id_csv = 'match&length.csv'          
output_fasta = 'selected_sequences.fasta'  

df_ids = pd.read_csv(id_csv)
selected_ids = set(df_ids['Sequence_ID'].astype(str)) 

print(f"共读取 {len(selected_ids)} 个筛选后的序列 ID")

def extract_fasta_by_ids(fasta_file, id_set, output_file):
    with open(fasta_file) as f_in, open(output_file, 'w') as f_out:
        write_flag = False
        seq_id = ''
        seq_lines = []

        for line in f_in:
            line = line.rstrip()
            if line.startswith('>'):
                if write_flag and seq_lines:
                    f_out.write(f">{seq_id}\n")
                    f_out.write("\n".join(seq_lines) + "\n")
                seq_id = line[1:].split()[0]  
                write_flag = seq_id in id_set
                seq_lines = []
            else:
                if write_flag:
                    seq_lines.append(line)
        if write_flag and seq_lines:
            f_out.write(f">{seq_id}\n")
            f_out.write("\n".join(seq_lines) + "\n")

extract_fasta_by_ids(fragments_fasta, selected_ids, output_fasta)

print(f"提取完成，已生成 {output_fasta}")

Extract the last 5 bp of the sequences and output them as a new FASTA file

In [ ]:
input_fasta = 'selected_sequences.fasta'   
output_fasta = 'last10bp_sequences_forward.fasta' 
bp_len = 5                                

with open(input_fasta) as f_in, open(output_fasta, 'w') as f_out:
    seq_id = ''
    seq_lines = []

    for line in f_in:
        line = line.rstrip()
        if line.startswith('>'):
            if seq_id and seq_lines:
                full_seq = ''.join(seq_lines)
                subseq = full_seq[-10:] if len(full_seq) >= bp_len else full_seq
                f_out.write(f">{seq_id}\n{subseq}\n")
            seq_id = line[1:].split()[0]  
            seq_lines = []
        else:
            seq_lines.append(line)
    if seq_id and seq_lines:
        full_seq = ''.join(seq_lines)
        subseq = full_seq[-bp_len:] if len(full_seq) >= bp_len else full_seq
        f_out.write(f">{seq_id}\n{subseq}\n")

print(f"已生成 {output_fasta}，每条序列包含最后 {bp_len} 个碱基")

Perform fragment filtering and extraction following the same workflow, using the reverse-complement of the template plasmid as the reference

Combine the extracted forward and reverse FASTA sequences into a single file

In [ ]:
file1 = "last10bp_sequences_forward.fasta"      
file2 = "last10bp_sequences_reverse.fasta"      
output_file = "merged.fasta"  


def merge_fasta(file1, file2, output_file):
    """将两个FASTA文件的内容顺序合并到一个文件中"""
    with open(output_file, 'w') as out_f:
        for infile in [file1, file2]:
            with open(infile, 'r') as f:
                for line in f:
                    out_f.write(line)
    print(f"已成功合并为 {output_file}")


merge_fasta(file1, file2, output_file)

Create a sequence logo from the last 10 bp of all extracted sequences and generate the associated information file

In [ ]:
import logomaker
from math import log2

In [ ]:
fasta_file = "merged.fasta"

def read_fasta(fasta_file):
    seqs = []
    with open(fasta_file, "r") as f:
        cur = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur:
                    seqs.append("".join(cur).upper())
                    cur = []
                continue
            cur.append(line)
        if cur:
            seqs.append("".join(cur).upper())
    return seqs

def build_frequency_matrix(seqs):
    bases = ["A", "C", "G", "T"]
    L = len(seqs[0])
    N = len(seqs)
    df = pd.DataFrame(0, index=range(L), columns=bases, dtype=float)
    for s in seqs:
        for i, b in enumerate(s):
            if b in bases:
                df.at[i, b] += 1
    df = df / N
    return df

def build_information_matrix(freq_df):
    df = freq_df.copy()
    info_df = pd.DataFrame(0, index=freq_df.index, columns=freq_df.columns, dtype=float)
    for i in freq_df.index:
        H = -sum(p * log2(p) for p in freq_df.loc[i] if p > 0)
        R = 2 - H  # DNA 最大信息量为 2 bits
        info_df.loc[i] = freq_df.loc[i] * R
    return info_df

def plot_logo(df, outname, ylabel):
    logo = logomaker.Logo(df,
                          font_name='Arial Rounded MT Bold',
                          color_scheme='classic')
    logo.style_spines(visible=False)
    logo.style_spines(spines=['left', 'bottom'], visible=True)
    logo.style_xticks(rotation=90)
    logo.ax.set_ylabel(ylabel)
    logo.fig.savefig(outname, dpi=300, bbox_inches='tight')
    plt.close(logo.fig)

if __name__ == "__main__":
    import matplotlib.pyplot as plt

    fasta_file = find_fasta_file(".")
    seqs = read_fasta(fasta_file)
    if len(set(len(s) for s in seqs)) != 1:
        raise ValueError("所有序列长度必须一致！")

    freq_df = build_frequency_matrix(seqs)
    freq_df.to_csv("frequency_matrix.csv", index_label="Position")
    print("✅ 已保存 frequency_matrix.csv")

    info_df = build_information_matrix(freq_df)
    info_df.to_csv("information_matrix.csv", index_label="Position")
    print("✅ 已保存 information_matrix.csv")

    plot_logo(freq_df, "logo_frequency.png", "Frequency")
    print("✅ 频率矩阵 logo 已生成：logo_frequency.png")

    plot_logo(info_df, "logo_information.png", "Information (bits)")
    print("✅ 信息量矩阵 logo 已生成：logo_information.png")